# Exploração dos dados do Zenodo (ADA RH Guaíba — Maio/2024)

Verificação do arquivo  (versão v0.4) gerado a partir do Zenodo e copiado para .
O arquivo consolida o impacto observado das inundações de maio de 2024 em 139 municípios da Região Hidrográfica do Guaíba, incluindo população, domicílios, área afetada (total e urbana), vias e estruturas públicas.

In [4]:
from pathlib import Path

import pandas as pd
import plotly.express as px

PASTA = Path("..", "..") / "dados" / "processados" / "zenodo"

df = pd.read_csv(PASTA / "analise_ada_rhguaiba_municipios.csv", dtype={"cd_mun": str})
print(f"Municípios: {len(df)} | Colunas: {len(df.columns)}")
df.head()

Municípios: 139 | Colunas: 26


,cd_mun,nm_mun,pessoas_afetadas,domicilios_afetados,pessoas_pct,domicilios_pct,pessoas_2022,domicilios_2022,vias_diretamente_afetadas_m,vias_diretamente_afetadas_km,...,area_diretamente_afetada_urbana_pct,estruturas_afetadas_nro,area_construida_afetada_m2,domicilios_particulares_afetados_diretamente,domicilios_coletivos_afetados_diretamente,estabelecim_agropecuario_afetados_diretamente,estabelecim_ensino_afetados_diretamente,estabelecim_saude_afetados_diretamente,edificacao_construcao_afetados_diretamente,estabelecim_religiosos_afetados_diretamente
0,4304606,Canoas,154002,65915,44.3,42.4,347657,155478,409744.0,409.7,...,39.43,89427.0,1.318513e+07,63788.0,30.0,15.0,103.0,52.0,1346.0,282.0
1,4314902,Porto Alegre,151141,85368,11.3,12.4,1332845,687679,518982.0,519.0,...,21.19,84523.0,1.733722e+07,91915.0,185.0,11.0,231.0,262.0,1166.0,298.0
2,4318705,São Leopoldo,83965,38349,38.6,39.0,217409,98238,269142.0,269.1,...,30.64,44298.0,6.184422e+09,34400.0,10.0,11.0,45.0,74.0,557.0,181.0
3,4306767,Eldorado do Sul,31761,13251,80.3,70.5,39559,18785,152201.0,152.2,...,86.36,22462.0,2.901761e+09,13385.0,6.0,82.0,37.0,21.0,307.0,53.0
4,4309308,Guaíba,20979,9109,22.6,22.2,92924,41073,62836.0,62.8,...,15.51,11077.0,1.638921e+09,8869.0,0.0,8.0,10.0,11.0,86.0,42.0


## Visão geral e totalizadores da bacia hidrográfica

Total de pessoas e domicílios atingidos na RH do Guaíba durante a enchente de maio de 2024.

In [5]:
tot_afetados = df["pessoas_afetadas"].astype(int).sum()
tot_pop_2022 = df["pessoas_2022"].astype(int).sum()
tot_dom_afetados = df["domicilios_afetados"].astype(int).sum()
tot_dom_2022 = df["domicilios_2022"].astype(int).sum()

print(f"População total da bacia (Censo 2022): {tot_pop_2022:,.0f}".replace(",", "."))
print(f"Pessoas diretamente afetadas: {tot_afetados:,.0f} ({tot_afetados/tot_pop_2022:.1%})".replace(",", "."))
print(f"Domicílios diretamente afetados: {tot_dom_afetados:,.0f} ({tot_dom_afetados/tot_dom_2022:.1%})".replace(",", "."))

População total da bacia (Censo 2022): 5.822.543
Pessoas diretamente afetadas: 632.669 (10.9%)
Domicílios diretamente afetados: 301.738 (10.9%)


## Top 15 municípios com mais pessoas afetadas

In [6]:
top15 = df.sort_values("pessoas_afetadas", ascending=False).head(15)

fig = px.bar(top15.sort_values("pessoas_afetadas"),
             x="pessoas_afetadas",
             y="nm_mun",
             orientation="h",
             text="pessoas_afetadas",
             labels={"pessoas_afetadas": "Pessoas afetadas", "nm_mun": "Município"},
             title="Top 15 municípios por pessoas diretamente afetadas (RH Guaíba)")
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=500)
fig.show()

**Comentário.** Canoas (154.002 pessoas afetadas, 44,3% da população) e Porto Alegre (151.141 pessoas afetadas, 11,3%) lideram o impacto absoluto, seguidas por São Leopoldo (83.965, 38,6%) e Eldorado do Sul (31.761, 80,3% da cidade).

## Municípios com maior percentual da população atingida

In [7]:
top_pct = df.sort_values("pessoas_pct", ascending=False).head(15)

fig = px.bar(top_pct.sort_values("pessoas_pct"),
             x="pessoas_pct",
             y="nm_mun",
             orientation="h",
             text="pessoas_pct",
             labels={"pessoas_pct": "População afetada (%)", "nm_mun": "Município"},
             title="Top 15 municípios com maior percentual de população afetada")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=500)
fig.show()

## Recorte de Porto Alegre: Comparação com as bases do INCT e IBGE

Verificação da coerência dos totais populacionais e indicadores municipais.

In [8]:
poa = df[df["cd_mun"] == "4314902"].iloc[0]

print("=== Porto Alegre (ADA RH Guaíba — Zenodo) ===")
print(f"População 2022: {int(poa["pessoas_2022"]):,} hab".replace(",", "."))
print(f"Pessoas afetadas: {int(poa["pessoas_afetadas"]):,} hab ({poa["pessoas_pct"]}%".replace(",", ".") + ")")
print(f"Domicílios afetados: {int(poa["domicilios_afetados"]):,} ({poa["domicilios_pct"]}%".replace(",", ".") + ")")
print(f"Área urbana atingida: {float(poa["area_diretamente_afetada_urbana_m2"])/1e6:.2f} km² ({poa["area_diretamente_afetada_urbana_pct"]}%)")
print(f"Vias afetadas: {float(poa["vias_diretamente_afetadas_km"]):.1f} km")
print(f"Estruturas afetadas: {int(poa["estruturas_afetadas_nro"]):,}".replace(",", "."))
print(f"Equipamentos de saúde atingidos: {poa["estabelecim_saude_afetados_diretamente"]}")
print(f"Equipamentos de ensino atingidos: {poa["estabelecim_ensino_afetados_diretamente"]}")
print(f"Estabelecimentos religiosos atingidos: {poa["estabelecim_religiosos_afetados_diretamente"]}")

=== Porto Alegre (ADA RH Guaíba — Zenodo) ===
População 2022: 1.332.845 hab
Pessoas afetadas: 151.141 hab (11.3%)
Domicílios afetados: 85.368 (12.4%)
Área urbana atingida: 37.19 km² (21.19%)
Vias afetadas: 519.0 km
Estruturas afetadas: 84.523
Equipamentos de saúde atingidos: 262.0
Equipamentos de ensino atingidos: 231.0
Estabelecimentos religiosos atingidos: 298.0


**Conclusão.** A população total de 1.332.845 habitantes de Porto Alegre é rigorosamente a mesma encontrada na base de setores censitários do IBGE e nos dados do INCT SiMAI. Isso valida o uso conjunto das três bases no projeto, permitindo cruzar os dados do INCT com as perdas mapeadas em maio de 2024.